# NVIDIA Stock Price Trend Analysis

## 基于调整后收盘价与移动平均线的趋势分析报告

本报告以 NVIDIA Corporation（NVDA）近五年股票价格数据为研究对象，使用 Pandas 对原始 CSV 数据进行清洗，并基于调整后收盘价构建 20 日与 60 日移动平均线，用于观察 NVDA 的中短期价格趋势。

报告重点关注三个问题：

1. NVDA 过去几年股价趋势如何？
2. MA20 与 MA60 如何反映短期和中期趋势？
3. NVDA 的上涨趋势是否伴随明显波动风险？


## 1. 数据来源与字段说明

本报告使用的原始数据文件为：

```text
D:\工作\七月份股票学习，以及相关作业完成\CSV\NVDA.csv
```

数据字段包括：

| 字段 | 含义 |
|---|---|
| Date | 交易日期 |
| Open | 当日开盘价 |
| High | 当日最高价 |
| Low | 当日最低价 |
| Close | 当日收盘价 |
| Adj_Close | 调整后收盘价，考虑分红、拆股等影响 |
| Volume | 当日成交量 |

本报告主要使用 `Adj_Close` 作为价格分析基础，因为它更适合进行长期价格趋势比较。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

file_path = r"D:\工作\七月份股票学习，以及相关作业完成\CSV\NVDA.csv"

df = pd.read_csv(file_path)

print("Raw data shape:", df.shape)
print("Raw columns:", df.columns.tolist())
df.head()

## 2. 数据清洗方法

原始 CSV 存在两个常见问题：

1. 列名中存在隐藏空格，例如 `\xa0`。
2. `Adj Close` 列在读取后可能显示为 `Unnamed: 5`。

因此，本报告先对列名进行标准化处理，并将异常列名修复为 `Adj_Close`。


In [ ]:
df.columns = (
    df.columns
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

if "Unnamed:_5" in df.columns:
    df = df.rename(columns={"Unnamed:_5": "Adj_Close"})

print("Cleaned columns:", df.columns.tolist())

此外，原始股票数据中可能包含 `Dividend` 或 `Stock Split` 等非正常交易行。这类记录不是每日价格数据，如果不删除，会影响数值转换和后续画图。


In [ ]:
rows_before = len(df)

mask = df.astype(str).apply(
    lambda row: row.str.contains("Dividend|Stock Split", case=False, na=False).any(),
    axis=1
)

df = df[~mask].copy()

rows_after = len(df)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)

完成异常行处理后，将日期字段转换为标准日期格式，并将价格与成交量字段转换为数值类型。


In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
df = df.dropna(subset=["Date"])

num_cols = ["Open", "High", "Low", "Close", "Adj_Close", "Volume"]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Adj_Close"])

print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

## 3. 样本区间与指标构建

为突出近五年趋势，本报告保留 2021 年之后的数据，并按日期升序排列。

构建的主要指标包括：

| 指标 | 含义 |
|---|---|
| MA20 | 20 日移动平均线，反映约一个月的短期趋势 |
| MA60 | 60 日移动平均线，反映约三个月的中期趋势 |

移动平均线可以帮助过滤短期噪音，更清楚地观察价格趋势。


In [ ]:
df = df.sort_values("Date")
df = df[df["Date"] >= "2021-01-01"]

df["MA20"] = df["Adj_Close"].rolling(20).mean()
df["MA60"] = df["Adj_Close"].rolling(60).mean()

print("Analysis period:")
print(df["Date"].min(), "to", df["Date"].max())

df[["Date", "Adj_Close", "MA20", "MA60"]].tail(10)

## 4. 股价趋势图

下图展示了 NVDA 调整后收盘价以及 20 日、60 日移动平均线。


In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Adj_Close"], label="Adj Close", linewidth=1.8)
plt.plot(df["Date"], df["MA20"], label="MA20", linewidth=1.5)
plt.plot(df["Date"], df["MA60"], label="MA60", linewidth=1.5)

plt.title("NVDA Adjusted Close Price with 20D and 60D Moving Averages", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Price (USD)")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 图表分析

从图中可以观察到以下几点：

### 5.1 2023 年后趋势明显增强

NVDA 股价在 2023 年后进入明显上升阶段，价格中枢快速抬升。这一阶段与 AI 基础设施需求增长、数据中心业务扩张以及市场对 GPU 需求的重新定价有关。

### 5.2 2024 年后中期趋势较强

2024 年之后，NVDA 股价大多数时间位于 MA60 上方，说明中期趋势整体偏强。MA20 也多数位于 MA60 上方，显示短期趋势持续强于中期趋势。

### 5.3 高收益伴随高波动

虽然长期趋势向上，但图中也可以看到多次明显回撤。尤其在价格快速上涨后，短期调整幅度也较大，说明 NVDA 并不是低波动股票。

### 5.4 MA20 与 MA60 的含义

MA20 更贴近股价，反应较快，适合观察短期趋势变化。MA60 更平滑，反应较慢，更适合观察中期趋势。两者结合可以帮助判断股价是否处于较强趋势中。


## 6. 结论

基于调整后收盘价和移动平均线的观察，NVDA 在 2021 至 2026 年间整体呈现显著上升趋势，尤其在 2023 年后上涨趋势明显增强。

从趋势角度看，2024 年后股价大部分时间位于 MA60 上方，说明 NVDA 中期走势较强。但与此同时，股价在上涨过程中也出现多次明显回撤，表明其收益弹性和波动风险都较高。

因此，NVDA 可以被视为强趋势、高波动的成长型股票。后续如果进一步做量化预测或策略回测，应同时关注收益、波动率、最大回撤和市场因子，而不能只看股价涨幅。


## 7. 局限性与后续改进

本报告属于基础价格趋势分析，仍有以下局限：

1. 只分析了价格和移动平均线，没有加入财报、估值和行业因子。
2. 移动平均线是滞后指标，只能反映历史趋势，不能直接预测未来价格。
3. 没有计算收益率、波动率、最大回撤等风险指标。
4. 没有与 SPY、QQQ、SOXX 等市场基准进行对比。

后续可以继续扩展：

- 加入日收益率和累计收益率。
- 计算 20 日波动率和最大回撤。
- 将 NVDA 与 SPY、QQQ、SOXX 进行对比。
- 加入财报事件和新闻情绪变量。
- 构建机器学习模型预测未来收益方向。
